# 🔧 LLM 3권 · 도구 (MCP · 함수 호출 · async · decorator) 종합 정리 노트

> **생성형 AI 기반 음성 에이전트 개발 과정 · LLM 챕터 3/3 — LLM에 손을 달아주는 도구 리뷰**
> `LLM/` 폴더 실습 노트북 24개를 **3권 체계**로 정리한 복습·재사용용 노트의 **3권**입니다.

| 항목 | 내용 |
|---|---|
| 이 권의 대상 | **도구 7종** — async/await · decorator · MCP 초급/실전 · 함수 호출 · Gmail-MCP (+ `MCP_이해하기.md`) |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | **macOS 전부 실행 가능** — 별도 서버/API 키 없이 미니 구현으로 재현 |
| 나머지 권 | **1권=사고부**(Thinking) · **2권=구조**(Attention·옴니·파인튜닝) |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **모두 macOS에서 실제 실행**됩니다.
> 진짜 MCP 클라이언트/서버(Gemini·Gmail)는 키·인증이 필요한 대신, **미니 이벤트 루프·미니 MCP·모의 우편함**으로
> 같은 원리를 재현합니다. 실물 호출부는 시그니처+가이드로 요약했습니다 (📄).


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 — 도구 7종 지도 · macOS 실행 판정 |
| **1** | 실험에 필요한 선행 지식 (async·decorator·MCP·함수호출) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (미니 구현·실물 연동 + macOS 가이드) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 도구 7종 한눈에

| 노트북 | 주제 | **M4 Pro 판정** | 핵심 포인트 |
|---|---|---|---|
| async/await | 이벤트 루프 해부 | ✅ 전부 실행 | 미니 루프·토큰 스트림·취소 정리 |
| decorator | `@` 해부 | ✅ 전부 실행 | `@mcp.tool()` 완전 해부 |
| MCP_Beginner | MCP 첫 실습 | ✅ 전부 실행 | 도구 등록·list/call |
| MCP_Real_API | 실전 연동 | ⚠️ 키/인증 필요 | Google Calendar·Gmail·Weather |
| gemini_function_calling | Gemini 함수 호출 | ✅ (키 있으면 실호출) | agent_loop·도구 선택 |
| gemini_mcp_gmail | Gemini×MCP | ⚠️ 키/인증 필요 | MCP 도구→함수 선언 번역 |
| MCP_이해하기.md | 개념 정리 | ✅ | 도구·자원·프롬프트 3요소 |

> **이 권의 흐름**: 파이썬 기초(async·decorator) → MCP 프로토콜 → LLM이 도구를 부르는 루프.
> **공통 뼈대**: `@tool()`로 등록 → `list_tools()`로 카탈로그 → `call_tool()`로 실행 → `isError`로 실패 회신.


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식 → ③ 2장 함수 실행하며
> "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. 파이썬 기초 — "LLM 도구의 재료"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 이벤트 루프 | 대기 중인 작업을 번갈아 실행하는 스케줄러 | async의 심장 |
| 코루틴 | `async def`로 만든 '중단 가능한 함수' | 정의만으론 실행 안 됨 — await가 시작 |
| await | "기다림을 루프에 알리고 양보" | time.sleep과 반대(블로킹) |
| 양보 (yield) | 실행 권한을 루프에게 넘김 | 미니 루프에서 `yield ("sleep", sec)` |
| async generator | 토큰을 하나씩 흘리는 생성기 | LLM 스트리밍의 원형 |
| 취소 (CancelledError) | 작업 중단 신호 | 정리 후 반드시 raise(삼키지 말 것) |
| decorator | 함수를 감싸는 장식 | 시간·재시도·등록 같은 교차 관심사 |
| functools.wraps | wrapper에 원본의 신분(__name__·__doc__) 복사 | 디버깅·문서화 보존 |
| 클로저 | 바깥 함수의 변수를 기억하는 함수 | 팩토리(decorator 공장)의 원리 |

### B. MCP — "LLM에게 도구를 표준으로 연결"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| MCP (Model Context Protocol) | LLM과 도구 서버를 연결하는 표준 규약 | 자작 JSON 대신 프로토콜 |
| @tool() | 함수를 '도구'로 등록 | 시그니처 → inputSchema 자동 파생 |
| inputSchema | 도구 인자의 JSON 스키마 | LLM이 "어떤 인자를" 알게 함 |
| list_tools | 서버가 가진 도구 목록 | 카탈로그 |
| call_tool | 등록된 도구 실행 | 인자 검증 후 호출 |
| isError | 도구 실패를 '결과'로 회신 | 예외를 삼키지 않고 모델에게 전달 |
| stdio_client | 표준 입출력으로 서버와 통신 | 별도 서버 프로세스 연결 |

### C. 함수 호출 — "LLM이 도구를 부르는 루프"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| function_call | 모델의 "이 도구를 불러라" 요청 | agent_loop의 시작 |
| function_response | 도구 실행 결과를 모델에게 회신 | 다음 행동 결정 재료 |
| agent_loop | 부를 때까지 ①~④ 반복 | 무한 루프 가드 max_steps |
| 구조화 오류 | 예외 대신 {"error": ...} 반환 | 도구 실패가 대화 종료가 아님 |

### D. Gmail-MCP
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 출력 계약 | GMAIL_RECORD_KEYS 7필드 | mock/real 모두 이 형태를 지켜야 |
| mock 우편함 | 실제 Gmail 대신 가짜 메일 데이터 | 키 없이 로직 검증 |
| 정규화 (normalize) | Gmail 원시 응답 → 공통 계약 형태 | mock이 미러하는 대상 |
| errlog | 자식 서버의 stderr를 받는 파일 | Jupyter의 stderr는 fileno가 없어 죽음 |

### E. 배경 지식 — 이 챕터가 왜 존재하는가
1·2권이 LLM의 '생각'과 '구조'라면, **3권은 "LLM에 손을 달아주는"** 도구 계층입니다.
1. **async/decorator** — 음성 에이전트에서 모든 것은 비동기(스트리밍)·반복(재시도)·등록(@tool)의 언어로 쓰인다.
2. **MCP/함수 호출** — LLM이 "날씨를 조회"하거나 "메일을 읽는" 행동을 하려면 도구가 필요하다.
   이 루프(`function_call → 실행 → function_response`)는 로컬이든 MCP 서버든 **실행부 한 줄만 다르다**.
3. **도구 실패의 철학** — 예외를 삼키지 말고 `isError`로 모델에게 회신. "오류도 대화의 일부"다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `mini_run`·`SleepOp`·`token_stream`·`with_cleanup` | 미니 이벤트 루프 | 양보·취소 |
| 2.1 | `timed`·`retry`·`register`·`MiniMCP_demo` | decorator 4종 | wraps·팩토리·등록·괄호 방어 |
| 2.2 | `MiniMCP.tool`·`list_tools`·`call_tool` | 미니 MCP | @tool → call + isError |
| 2.3 | `MockAgentLLM`·`agent_loop` | 에이전트 루프 | function_call → response |
| 2.4 | `_validate_record`·`list_messages`·`search_messages` | Gmail mock | 출력 계약 |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. async/await — "기다림"을 루프에 알리고 양보한다

- `await asyncio.sleep(0.5)` = 기다림을 루프에 알리고 **제어권 반환**. `time.sleep`은 블로킹(루프 정지).
- **코루틴 = 정의일 뿐** — 실행은 `asyncio.run`·`await`가 시작한다.
- **취소**: `CancelledError`를 받으면 자원 정리 후 **반드시 raise** (삼키면 호출자가 취소 사실을 모른다).
- **미니 루프**: `yield ("sleep", sec)` — awaitable이 루프에게 요청을 '양보'로 전달한다 (내부 동작 해부).
- **async generator**: `yield`로 토큰을 흘린다 — LLM 스트리밍 응답의 원형.

## 1-2. decorator — 함수를 감싸고, 신분을 보존한다

- `functools.wraps(fn)` — wrapper에 fn의 `__name__`·`__doc__` 복사 (신분 보존).
- **팩토리 3층**: `@retry(times)` → `retry`는 설정을 받는 공장 → 진짜 decorator → 실행 대리인 wrapper.
- **등록 패턴**: `@register(intent)`는 감싸지 않고 **명부에 올린 뒤 원본을 그대로 반환** — `HANDLERS[intent] = fn`.
- `@mcp.tool()`의 괄호 실수 방어: `if callable(name): raise TypeError` — 정의 시점에 잡는다.

## 1-3. MCP (Model Context Protocol) — LLM에게 도구를 표준으로 연결

```
@tool()으로 함수 등록 → inputSchema(JSON 스키마) 자동 생성
  → list_tools()로 모델에게 카탈로그 제공 → 모델이 call_tool(name, args) 요청
  → 서버 실행 → CallToolResult(isError 포함)로 회신
```

- **도구·자원·프롬프트 3요소** (MCP_이해하기.md): 도구=행동, 자원=데이터, 프롬프트=지시.
- **표준 전송**: `stdio_client` + `ClientSession` — initialize → list_tools → call_tool.
- **V2 관찰**: `Tool`이 `inputSchema`(snake_case)로 변경 — 버전 파괴적 변경에 주의.

## 1-4. 함수 호출 (Function Calling) — LLM이 도구를 부르는 루프

```
① 모델이 function_call 요청 → ② 앱이 도구 실행 → ③ 결과를 function_response로 회신
  → ④ 모델이 더 부르지 않을 때까지 반복 (무한 루프 가드 max_steps)
```

- 예외도 `{"error": ...}`로 구조화해 **모델에게 회신** (V4 관례) — 도구 실패가 곧 대화 종료가 아니다.
- `agent_loop`와 `mcp_agent_loop`는 **실행부 한 줄만 다르다** — 도구 소스가 로컬이냐 MCP 서버냐.

## 1-5. Gmail-MCP — mock과 real의 이중 계약

- **출력 계약**: `GMAIL_RECORD_KEYS`(id/from_addr/subject/date/snippet/body/labels) — mock이든 real이든
  이 형태를 벗어나면 `_validate_record`가 시끄럽게 죽인다. mock이 미러하는 대상은 real의 정규화 출력.
- **MCP 관례**: 도구 내 예외는 `isError=True`인 CallToolResult로 모델에게 회신 (삼키지 않는다).
- **V10 교훈**: Jupyter의 stderr는 파일이 아니어서 자식 프로세스 로그가 죽는다 → `errlog`에 **진짜 파일** 전달.


# 2. 함수/클래스 정의 및 주석 🔧

> ✅ = 실행 코드 셀 (macOS에서 실제 실행) · 📄 = 요약만 (실물 API·인증은 3장 가이드)


In [ ]:
# ═══ 2.0 async/await — 미니 이벤트 루프 · 토큰 스트림 · 취소 정리 ✅ ═══
# ▶ 미니 이벤트 루프: SleepOp가 yield("sleep", sec)로 요청을 전달 — await의 내부 동작을 해부.
#   token_stream: async generator가 LLM 스트리밍의 원형. with_cleanup은 취소를 삼키지 않는다.
import asyncio, time, heapq
try:
    import nest_asyncio
    nest_asyncio.apply()          # ipykernel(돌고 있는 루프)에서 asyncio.run 허용
except ImportError:
    pass

# ① 코루틴은 정의일 뿐 — await가 실행을 시작한다
async def hello():
    return 42
assert asyncio.run(hello()) == 42

# ② 미니 이벤트 루프 — 우리만의 awaitable이 루프에 요청을 '양보'로 전달한다
class SleepOp:
    def __init__(self, sec):
        self.sec = sec
    def __await__(self):
        yield ("sleep", self.sec)         # 루프에게 "이 초만큼 재워달라"는 요청

def mini_sleep(sec):
    return SleepOp(sec)

def mini_run(*coros):
    # 단일 스레드 스케줄러: 준비된 코루틴을 꺼내 다음 양보 지점까지 민다
    ready = list(coros)
    sleeping = []                         # (깨울 시각, 순번, 코루틴)
    seq = 0
    while ready or sleeping:
        if not ready:                     # 전부 자는 중 → 최초 기상 시각까지 실제 대기
            wake_at, _, coro = heapq.heappop(sleeping)
            time.sleep(max(0.0, wake_at - time.monotonic()))
            ready.append(coro)
        coro = ready.pop(0)
        try:
            op, arg = coro.send(None)     # 다음 양보 지점까지 실행
        except StopIteration:
            continue
        if op == "sleep":
            seq += 1
            heapq.heappush(sleeping, (time.monotonic() + arg, seq, coro))

order = []
async def job(name, sec):
    order.append(f"{name}:시작")
    await mini_sleep(sec)                 # 우리 루프 위에서 동작하는 await
    order.append(f"{name}:재개")

mini_run(job("A", 0.3), job("B", 0.1))    # B(0.1)이 A(0.3)보다 먼저 깬다
assert order[0].startswith("A") and order[1].startswith("B")
assert order[2].startswith("B") and order[3].startswith("A")

# ③ 토큰 스트림 — async generator는 LLM 스트리밍의 원형
async def token_stream():
    for tok in ["안녕", "하세요", ",", " 고객님", "."]:
        await asyncio.sleep(0)
        yield tok
async def collect():
    return [t async for t in token_stream()]
assert asyncio.run(collect()) == ["안녕", "하세요", ",", " 고객님", "."]

# ④ 취소 정리 — CancelledError는 정리 후 반드시 raise
cleaned = []
async def with_cleanup():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        cleaned.append(True)              # 자원 정리(연결 닫기 등)
        raise                             # 삼키지 말 것
async def cancel_demo():
    t = asyncio.create_task(with_cleanup())
    await asyncio.sleep(0)
    t.cancel()
    try:
        await t
    except asyncio.CancelledError:
        pass
asyncio.run(cancel_demo())
assert cleaned == [True]
print("async/await 검증 통과 ✅ — 미니 루프 스케줄링 / 토큰 스트림 / 취소 정리")



In [ ]:
# ═══ 2.1 decorator — timed · retry · 등록 패턴 · @tool() 괄호 방어 ✅ ═══
# ▶ decorator 4종: timed(시간) · retry(재시도) · register(등록, 감싸지 않음) · @tool() 괄호 방어.
#   functools.wraps가 신분(__name__·__doc__)을 보존 — 감싼 함수를 잃지 않는다.
import functools, inspect, time

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        wrapper.last_ms = (time.perf_counter() - t0) * 1000
        return result
    return wrapper

def retry(times):
    # 1층: 설정을 받는 공장 → 2층: 진짜 데코레이터 → 3층: 실행 대리인
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    if attempt == times:
                        raise
        return wrapper
    return decorator

calls = []
@retry(3)
def flaky_api():
    calls.append(1)
    if len(calls) < 3:
        raise ConnectionError("일시 오류")
    return "응답 수신"

@timed
def run_llm(text):
    time.sleep(0.01)
    return {"intent": "refund"}

assert flaky_api() == "응답 수신" and len(calls) == 3
assert run_llm("환불 문의")["intent"] == "refund" and run_llm.last_ms > 0
assert run_llm.__name__ == "run_llm"          # functools.wraps가 신분을 보존

# 등록 패턴 — 감싸지 않고 명부에 올린 뒤 원본을 그대로 반환
HANDLERS = {}
def register(intent):
    def decorator(fn):
        HANDLERS[intent] = fn                 # ① 명부에 올리고
        return fn                             # ② 원본 그대로 (감싸지 않음)
    return decorator

@register("refund")
def handle_refund(msg):
    return f"환불 접수: {msg}"

@register("reserve")
def handle_reserve(msg):
    return f"예약 접수: {msg}"

assert set(HANDLERS) == {"refund", "reserve"}
assert HANDLERS["refund"]("A-001") == "환불 접수: A-001"

# @tool() 괄호 실수 방어 — 정의 시점에 잡는다
class MiniMCP_demo:
    def tool(self, name=None, description=None):
        if callable(name):                    # @tool(괄호 없음) 실수 → TypeError
            raise TypeError("@tool()처럼 괄호를 붙여 호출하십시오 (@tool 아님)")
        def decorator(fn):
            return fn
        return decorator

m2 = MiniMCP_demo()
try:
    @m2.tool
    def oops(x):
        return x
    raise AssertionError("괄호 실수 방어 미포착")
except TypeError:
    pass
print("decorator 검증 통과 ✅ — timed·wraps / retry 3회 / 등록 패턴 / @tool() 방어")


In [ ]:
# ═══ 2.2 미니 MCP — @tool 등록 · list_tools · call_tool ✅ ═══
# ▶ MiniMCP: @tool로 등록(감싸지 않고 명부에 저장) → list_tools(카탈로그) → call_tool(실행).
#   도구 실패는 예외가 아니라 {"isError": True, ...}로 회신 — MCP 관례.
import json

class MiniMCP:
    # MCP 서버의 심장 — 도구 등록(@tool)과 호출(call_tool)을 한 클래스에
    def __init__(self, name):
        self.name = name
        self.tools = {}
    def tool(self, name=None, description=None):
        if callable(name):                    # 괄호 실수는 정의 시점에 잡는다
            raise TypeError("@tool()처럼 괄호를 붙여 호출하십시오")
        def decorator(fn):
            tool_name = name or fn.__name__
            self.tools[tool_name] = {
                "fn": fn,
                "description": description or fn.__doc__ or "",
            }
            return fn                         # ★ 감싸지 않는다 (등록 패턴)
        return decorator
    def list_tools(self):
        return [{"name": k, "description": v["description"]}
                for k, v in self.tools.items()]
    def call_tool(self, name, arguments):
        if name not in self.tools:
            return {"isError": True, "text": f"미지의 도구: {name}"}
        try:
            out = self.tools[name]["fn"](**arguments)
            return {"isError": False, "text": out}
        except Exception as e:
            return {"isError": True, "text": str(e)}    # MCP 관례: 예외를 삼키지 않고 회신

mcp = MiniMCP("assistant")

@mcp.tool(description="도시의 현재 날씨를 조회한다")
def get_weather(city: str, unit: str = "celsius") -> str:
    return f"{city}/{unit}"

@mcp.tool(description="현재 시스템 시간을 반환한다")
def get_current_time(timezone: str = "KST") -> str:
    return f"현재 시간 (KST)"

@mcp.tool(description="최근 읽지 않은 이메일 목록을 가져온다")
def check_unread_emails(limit: int = 3) -> str:
    mock = [
        {"sender": "boss@company.com", "subject": "[긴급] 내일 회의 준비", "date": "오늘 09:00"},
        {"sender": "newsletter@ai.com", "subject": "이번 주 AI/MCP 소식", "date": "오늘 10:30"},
    ]
    return json.dumps(mock[:limit], ensure_ascii=False, indent=2)

names = {t["name"] for t in mcp.list_tools()}
assert names == {"get_weather", "get_current_time", "check_unread_emails"}
assert mcp.call_tool("get_weather", {"city": "서울"}) == {"isError": False, "text": "서울/celsius"}
assert mcp.call_tool("get_weather", {"city": "서울", "unit": "fahrenheit"})["text"] == "서울/fahrenheit"
assert json.loads(mcp.call_tool("check_unread_emails", {"limit": 1})["text"])[0]["sender"] == "boss@company.com"
assert mcp.call_tool("no_such", {})["isError"] is True        # 미지의 도구
assert mcp.call_tool("get_weather", {})["isError"] is True    # 필수 인자 누락 → 예외 회신
print("미니 MCP 검증 통과 ✅ — @tool 등록 / list_tools 카탈로그 / call_tool + isError 관례")


In [ ]:
# ═══ 2.3 함수 호출 에이전트 루프 — 모델이 도구를 부른다 ✅ ═══
# ▶ agent_loop: 모델이 도구를 부를 때까지 ①~④ 반복 (무한 루프 가드 max_steps).
#   실행부 한 줄(session.call_tool)이 로컬↔MCP 차이 — 나머지는 재사용.
class MockAgentLLM:
    # 모의 LLM: 도구 호출(function_call) 또는 최종 답변을 결정한다
    def generate(self, user_msg, contents):
        has_result = any(c["role"] == "function" for c in contents)
        if has_result:                                # 도구 결과를 받은 뒤 → 최종 답변
            last = [c for c in contents if c["role"] == "function"][-1]
            return {"call": None, "text": f"검색 결과를 안내드립니다: {last['result']}"}
        if "날씨" in user_msg:
            return {"call": {"name": "get_weather", "args": {"city": "서울"}}, "text": None}
        if "시간" in user_msg:
            return {"call": {"name": "get_current_time", "args": {}}, "text": None}
        return {"call": None, "text": "안녕하세요, 무엇을 도와드릴까요?"}

def agent_loop(user_msg, llm, mcp, max_steps=5):
    # 모델이 더 부르지 않을 때까지 ①~④를 반복한다 (무한 루프 가드)
    contents = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        out = llm.generate(user_msg, contents)
        if out["call"] is None:                       # 도구 요청 없음 → 최종 답변
            return out["text"], step
        name = out["call"]["name"]; args = out["call"]["args"]
        result = mcp.call_tool(name, args)            # ★ 실행부 — 여기만 소스가 갈린다
        if result["isError"]:
            contents.append({"role": "function", "name": name, "result": result["text"]})
        else:
            contents.append({"role": "function", "name": name, "result": result["text"]})
    return "(중단: 최대 스텝 초과 — 루프 가드 발동)", max_steps

llm = MockAgentLLM()

final, steps = agent_loop("서울 날씨 알려줘", llm, mcp)
assert "서울/celsius" in final and steps == 2          # 1단계 호출 + 2단계 답변

final2, steps2 = agent_loop("안녕하세요", llm, mcp)
assert steps2 == 1 and "무엇을 도와드릴까요" in final2   # 도구 없이 즉시 답변
print("함수 호출 검증 통과 ✅ — ①function_call → ②실행 → ③function_response → ④최종답변")


In [ ]:
# ▶ 데모 — '@tool()로 등록하면 무슨 일이 생기는지' 눈으로 확인 (초보자용)
# 함수를 정의한 뒤 @tool을 붙이는 것만으로 '도구'가 되는 흐름을 단계로 본다.

mcp = MiniMCP("assistant")

@mcp.tool(description="도시의 현재 날씨를 조회한다")
def get_weather(city: str, unit: str = "celsius") -> str:
    return f"{city}/{unit}"

@mcp.tool(description="현재 시스템 시간을 반환한다")
def get_current_time(timezone: str = "KST") -> str:
    return f"현재 시간 (KST)"

# ① list_tools: LLM이 보게 되는 '카탈로그' — 이름과 설명만 노출
print("① 카탈로그:")
for t in mcp.list_tools():
    print(f"   - {t['name']}: {t['description']}")

# ② call_tool: 모델이 골라 호출 — 인자 검증 후 실행
r = mcp.call_tool("get_weather", {"city": "서울", "unit": "fahrenheit"})
print(f"② 호출 결과: {r}")

# ③ 잘못된 인자 → isError=True로 회신 (예외로 터지지 않음)
bad = mcp.call_tool("get_weather", {})
print(f"③ 필수 인자 누락: isError={bad['isError']}")

assert {"get_weather", "get_current_time"} <= {t["name"] for t in mcp.list_tools()}
assert bad["isError"] is True
print("데모 통과 ✅ — @tool() 한 줄이 함수를 'LLM이 부를 수 있는 도구'로 승격시킨다")


In [ ]:
# ═══ 2.4 Gmail-MCP — 출력 계약 · mock 우편함 · 검색 ✅ ═══
# ▶ Gmail-MCP: mock/real 모두 GMAIL_RECORD_KEYS 7필드를 지킨다 — 출력 계약.
#   get_message는 미지의 id에 ValueError(삼키지 않음) — list로 먼저 확인하라는 안내 포함.
GMAIL_RECORD_KEYS = ("id", "from_addr", "subject", "date", "snippet", "body", "labels")

def _validate_record(rec):
    # 출력 계약 — mock이든 real이든 이 형태를 벗어나면 여기서 시끄럽게 죽는다
    missing = [k for k in GMAIL_RECORD_KEYS if k not in rec]
    assert not missing, f"GMAIL_RECORD_KEYS 계약 위반 — 누락 필드: {missing}"
    return rec

MOCK_MAILBOX = [
    {"id": "m1", "from_addr": "boss@company.com", "subject": "[긴급] 내일 회의 준비",
     "date": "오늘 09:00", "snippet": "회의 자료 준비 부탁드립니다.",
     "body": "회의 자료 준비 부탁드립니다.", "labels": ["INBOX", "UNREAD"]},
    {"id": "m2", "from_addr": "system@cloud.com", "subject": "서버 결제 영수증",
     "date": "어제 18:00", "snippet": "이번 달 청구 금액 안내입니다.",
     "body": "이번 달 청구 금액 안내입니다.", "labels": ["INBOX"]},
    {"id": "m3", "from_addr": "shop@mall.com", "subject": "환불 안내 (ORD-4821)",
     "date": "어제 21:00", "snippet": "환불이 완료되었습니다.",
     "body": "환불이 완료되었습니다.", "labels": ["INBOX", "UNREAD"]},
]

def list_messages(max_results=10, label="INBOX"):
    # 목록은 본문 제외 (본문은 get_message로)
    rows = [r for r in MOCK_MAILBOX if label in r["labels"]][:max_results]
    slim = [{k: r[k] for k in ("id", "from_addr", "subject", "date", "snippet")}
            for r in map(_validate_record, rows)]
    return json.dumps(slim, ensure_ascii=False)

def get_message(message_id):
    for r in MOCK_MAILBOX:
        if r["id"] == message_id:
            return json.dumps(_validate_record(r), ensure_ascii=False)
    raise ValueError(f"메일 없음: {message_id} — list_messages로 유효한 id를 먼저 확인하라")

def search_messages(query):
    # 제목·본문·발신자에서 키워드 검색
    q = query.lower()
    rows = [r for r in MOCK_MAILBOX
            if q in r["subject"].lower() or q in r["body"].lower() or q in r["from_addr"].lower()]
    slim = [{k: r[k] for k in ("id", "from_addr", "subject", "date", "snippet")}
            for r in map(_validate_record, rows)]
    return json.dumps(slim, ensure_ascii=False)

inbox = json.loads(list_messages())
assert len(inbox) == 3 and "body" not in inbox[0]         # 목록엔 본문 없음
unread = json.loads(list_messages(label="UNREAD"))
assert {r["id"] for r in unread} == {"m1", "m3"}

full = json.loads(get_message("m2"))
assert full["subject"] == "서버 결제 영수증" and full["body"]
try:
    get_message("없는_id")
    raise AssertionError("미지의 메일 미포착")
except ValueError:
    pass

hit = json.loads(search_messages("환불"))
assert len(hit) == 1 and hit[0]["id"] == "m3"
assert json.loads(search_messages("없는단어")) == []

try:
    _validate_record({"id": "x"})                        # 필수 키 누락
    raise AssertionError("출력 계약 위반 미포착")
except AssertionError:
    pass
print("Gmail-MCP 검증 통과 ✅ — 출력 계약 / 목록·본문·검색 / 미지의 메일·위반 시끄럽게 실패")


## 2.5 📄 요약 — 도구 7종 핵심 카드

| | async/await | decorator | MCP | 함수 호출 | Gmail-MCP |
|---|---|---|---|---|---|
| 핵심 | 양보·취소 | wraps·팩토리·등록 | @tool→call | agent_loop | 출력 계약 |
| macOS | ✅ | ✅ | ✅ 미니 재현 | ✅ 모의 | ✅ mock 우편함 |
| 실물 | — | — | stdio_client | Gemini 키 | Google 인증 |

**교훈 4줄**:
1. `await`는 "기다림을 알리고 양보" — `time.sleep`은 루프를 통째로 멈춘다.
2. `@mcp.tool()`은 감싸지 않고 **명부에 등록** — 괄호 실수는 정의 시점에 TypeError로 방어.
3. MCP·함수 호출 모두 **`function_call → 실행 → function_response` 루프** — 실행부 한 줄만 다르다.
4. 도구 실패는 예외를 삼키지 말고 `isError`로 모델에게 회신 — "오류도 대화의 일부".


## 2.6 [REAL] 실물 실행 — OpenAI function-calling 왕복

> 2.3의 에이전트 루프를 **실제 API**의 `tools` 계약으로 재현합니다. 준비: `.env` 에 `OPENAI_API_KEY`


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not (ae.has("openai") and ae.key_present("OPENAI")):
    print("OPENAI_API_KEY 없음 → 스킵 (.env 등록 후 커널 재시작)")
else:
    import openai, json
    client = openai.OpenAI(api_key=ae.KEYS["OPENAI"])
    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "지역의 현재 날씨를 조회한다",
            "parameters": {"type": "object",
                           "properties": {"city": {"type": "string"}},
                           "required": ["city"]},
        }}]
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "서울 날씨 알려줘"}],
        tools=tools, tool_choice="auto", max_tokens=128)
    msg = r.choices[0].message
    print("모델 답:", msg.content)
    print("tool_calls:", [tc.function.name for tc in (msg.tool_calls or [])])
    assert msg.tool_calls, "모델이 도구를 불러야 한다"
    args = json.loads(msg.tool_calls[0].function.arguments)
    result = {"city": args["city"], "weather": "맑음 22°C"}
    r2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "서울 날씨 알려줘"},
                  {"role": "assistant", "content": None, "tool_calls": msg.tool_calls},
                  {"role": "tool", "tool_call_id": msg.tool_calls[0].id,
                   "content": json.dumps(result, ensure_ascii=False)}],
        max_tokens=128)
    print("최종 응답:", r2.choices[0].message.content)
    print("function-calling 왕복 통과 ✅ (@tool 등록 → 호출 → 결과 회신)")


# 3. 실험 진행 방법 🧪

## 3-1. macOS에서 전부 실행 — 이 노트 셀 순서
```
2.0 미니 이벤트 루프 → 2.1 데코레이터 4종 → 2.2 미니 MCP(등록/호출)
→ 2.3 에이전트 루프 → 2.4 Gmail mock 우편함
```
별도 설치·키 없이 위에서 아래로 실행하면 됩니다.

## 3-2. 실물 연동 (선택 — 키/인증 필요)
| 실습 | 필요 | macOS 가이드 |
|---|---|---|
| MCP_Beginner | `mcp` 라이브러리 | `pip install mcp` 후 `run_real_world_test()` |
| gemini_function_calling | `GOOGLE_API_KEY` | `export GOOGLE_API_KEY=...` → `agent_loop` (1권 5-A 참조) |
| MCP_Real_API | Google OAuth + OpenWeather 키 | `credentials.json` + `OPENWEATHER_API_KEY` |
| gemini_mcp_gmail | Google OAuth (Gmail 읽기) | `token.json` 생성 후 `MODE=real` |

> **V10 핵심 주의**: 노트북에서 MCP 서버를 띄울 때 자식 프로세스 stderr는 **진짜 파일**로 전달 —
> `stdio_client(params, errlog=open(SERVER_LOG,"w"))` (fileno() 없는 노트북 stderr는 죽는다).

## 3-3. 판단 기준
1. **양보 여부**: `time.sleep`(블로킹) vs `await sleep`(양보) — 이벤트 루프 응답성으로 판정.
2. **신분 보존**: `functools.wraps` 유무 — `fn.__name__`·`__doc__` 유지 확인.
3. **등록 vs 감싸기**: `@tool()`이 원본을 그대로 반환하는지 (감싸면 시그니처/스키마가 깨진다).
4. **isError 관례**: 도구 실패가 예외로 터지지 않고 결과로 회신되는지.
5. **출력 계약**: mock/real 모두 `GMAIL_RECORD_KEYS`를 지키는지 (실제 Gmail이 어떤 형태를 주는지 먼저 관찰).


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. 도구 계층 구조

```
LLM (Gemini/Qwen…)          ← function_call 요청 / 최종 답변
  ↕  (도구 선언 = inputSchema)
에이전트 루프 (agent_loop)   ← function_response 회신 · max_steps 가드
  ↕  (call_tool, isError 관례)
MCP 클라이언트 (stdio_client + ClientSession)  ──  로컬 함수(@tool) 또는 원격 서버
```

## 4-2. @tool() 등록의 설계 (MiniMCP)
- **감싸지 않는다** — `fn`을 그대로 반환 (시그니처 보존) · 명부(`self.tools`)에 메타만 저장.
- **docstring = 도구 설명** · 시그니처 → inputSchema(JSON 스키마) 자동 파생.
- **괄호 실수 방어**는 정의 시점에 — "나중에"가 아니라 "바로" 실패한다.

## 4-3. 에이전트 루프 확장 포인트 (1권 5-T와 연결)
`agent_loop`(로컬 도구)와 `mcp_agent_loop`(MCP 서버)는 **실행부 한 줄만 다르다** —
도구 소스를 교체해도 루프·가드·오류 회신 구조는 재사용된다. 1권 5-T의 `route()`와 결합하면
"LLM 분류 → 결정적 게이트 → 도구 실행"으로 이어진다.

## 4-4. 최종 판정
- **기초**: async는 '양보'의 언어, decorator는 '교차 관심사(시간·재시도·등록)'의 문법.
- **표준**: 도구 연결은 자작 JSON 대신 **MCP**(프로토콜)로 — 클라이언트·서버·스키마가 표준화된다.
- **실패**: 도구 예외는 `isError`로 모델에게 회신 — 모델이 다음 행동을 정한다 (대화가 끊기지 않는다).
